# Full Dataset Replication v1.0.0

This notebook runs the preregistered ten-seed comparison of raw samples, centered Fourier features, and Fourier-plus-inferred-jump descriptors on **Synthetic Fourier Noise Dataset v1.0.0**. It is a 1024-point confirmatory extension of the historical 1500-point manuscript pilot, not a byte-for-byte numerical rerun. The frozen protocol is `experiments/full_dataset_replication_v1.0.0.json`.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/abbass12/FourierSeriesClassification.git'
DATASET_URL = 'https://files.manuscdn.com/user_upload_by_module/session_file/310419663029089887/ShRoOXiZbDwEtWwb.zip'
DATASET_SHA256 = 'a9f2f07b690ebef252bd0ce33214a80bee80ebcfe6b12b1e02599d2e2cfdef9e'
REPO_DIR = Path('/content/FourierSeriesClassification')
ARCHIVE = Path('/content/SyntheticFourierNoiseDataset_v1.0.0.zip')
DATASET_PARENT = Path('/content/dataset_release')
RESULT_DIR = Path('/content/full_dataset_replication_v1.0.0')
for path in (REPO_DIR, DATASET_PARENT, RESULT_DIR):
    if path.exists():
        shutil.rmtree(path)
if ARCHIVE.exists():
    ARCHIVE.unlink()
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Python:', sys.version.split()[0])

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a T4 GPU runtime in Runtime > Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
!curl -L --retry 3 --fail -o {ARCHIVE} {DATASET_URL}
!echo '{DATASET_SHA256}  {ARCHIVE}' | sha256sum -c -
!mkdir -p {DATASET_PARENT}
!unzip -q {ARCHIVE} -d {DATASET_PARENT}
!python dataset_generation/validate_synthetic_fourier_dataset.py --dataset-dir {DATASET_PARENT}/release_artifact

In [ ]:
!python run_full_dataset_replication.py \
  --dataset-dir {DATASET_PARENT}/release_artifact \
  --output-dir {RESULT_DIR} \
  --condition clean \
  --seeds 11 23 37 53 71 89 107 131 149 167 \
  --modes 50 --max-jumps 4 --epochs 80 --batch-size 256 --lr 0.001 --patience 12 --device cuda

In [ ]:
!python validation/validate_full_dataset_replication.py --result-dir {RESULT_DIR}
!cat {RESULT_DIR}/validation_report.md
!cat {RESULT_DIR}/summary.json
!du -sh {RESULT_DIR}

In [ ]:
ARCHIVE_RESULTS = Path('/content/full_dataset_replication_v1.0.0.zip')
if ARCHIVE_RESULTS.exists():
    ARCHIVE_RESULTS.unlink()
shutil.make_archive(str(ARCHIVE_RESULTS.with_suffix('')), 'zip', RESULT_DIR.parent, RESULT_DIR.name)
!sha256sum {ARCHIVE_RESULTS}
print('Result archive:', ARCHIVE_RESULTS)